# P-HBCC-2M — notebook thử nghiệm đã ngừng dùng làm pipeline chính

> P-HBCC-2M không còn thuộc bảng kết luận chính. Hãy dùng `notebooks/hbcc_fair_training.ipynb` để huấn luyện HBCC-Small/Medium cũ và các đối chứng với augmentation công bằng. File này chỉ được giữ để tái lập thử nghiệm kiến trúc mới; các con số 200 epoch/3 seed bên dưới là thiết kế lịch sử, không phải protocol hiện tại.

Notebook này chạy kiến trúc **P-HBCC-2M** và bộ đối chứng cốt lõi bằng **một protocol duy nhất** lấy cảm hứng từ Context Cluster (ICLR 2023, Mục 4.1): RandomCrop thích nghi cho CIFAR, horizontal flip, Random Erasing, MixUp, CutMix và label smoothing. RandAugment được bỏ để tránh thêm một augmentation không được liệt kê trong phần mô tả của bài báo. Hãy chọn kernel **Python (CoC)** hoặc interpreter `D:\Anaconda\envs\CoC\python.exe`.

Mặc định notebook chỉ chạy **smoke test một batch** cho P-HBCC-2M. Ma trận chính gồm 5 mô hình × 3 paired seeds × 200 epochs = 3.000 epoch-runs, giảm 75% so với thiết kế mở rộng 8 × 5 × 300. Muốn chạy đầy đủ, đặt `RUN_FULL = True` trong cell cấu hình. Kết quả rút gọn được gắn protocol không canonical và có hậu tố `_smoke`/`_eN`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Có thể gán đường dẫn thủ công nếu kernel mở notebook từ thư mục khác.
REPO_ROOT = None  # Ví dụ: r'D:\\Research\\Lightweight-Context-Cluster'

cwd = Path.cwd().resolve()
candidates = [Path(REPO_ROOT).resolve()] if REPO_ROOT else [cwd, *cwd.parents]
ROOT = next((p for p in candidates if (p / 'lightweight_hbcc').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Không tìm thấy thư mục gốc Lightweight-Context-Cluster.')

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PYTHON = sys.executable
print('Repository :', ROOT)
print('Python     :', PYTHON)


## 1. Cấu hình chạy

- `RUN_SMOKE=True`: chạy 1 epoch, mỗi split chỉ 1 batch.
- `FORCE_SMOKE=True`: chủ động ghi đè smoke output cũ; mặc định tắt để tránh mất kết quả.
- `RUN_FULL=True`: chạy 15 lượt (5 mô hình × 3 paired seeds), mỗi lượt 200 epoch; runner không ghi đè kết quả cũ.
- Đổi `DATASET` thành `cifar100` để chạy 100 lớp.

In [ ]:
from tools.run_fair_comparison import (
    CORE_MODELS as RUNNER_CORE_MODELS,
    DEFAULT_SEEDS as RUNNER_DEFAULT_SEEDS,
    MODEL_CONFIGS,
)

DATASET = 'cifar10'  # 'cifar10' hoặc 'cifar100'

AVAILABLE_MODELS = list(MODEL_CONFIGS)

CORE_MODELS = list(RUNNER_CORE_MODELS)

SMOKE_MODELS = ['phbcc_2m']
SMOKE_SEEDS = [17]
FULL_MODELS = CORE_MODELS
FULL_SEEDS = list(RUNNER_DEFAULT_SEEDS)

RUN_SMOKE = False  # Notebook lưu trữ: không chạy P-HBCC mặc định
FORCE_SMOKE = False
RUN_FULL = False  # Đổi thành True khi sẵn sàng chạy ma trận 200 epoch
RUN_BENCHMARK_AFTER_FULL = True

DATA_ROOT = ROOT / 'data'
OUTPUT_ROOT = ROOT / 'runs_fair_paper_inspired_200e_notebook' / DATASET
BENCHMARK_ROOT = ROOT / 'results' / 'fair_paper_inspired_200e_notebook'

assert DATASET in {'cifar10', 'cifar100'}
print('Dataset     :', DATASET)
print('Smoke models:', SMOKE_MODELS)
print('Full models :', FULL_MODELS)
print('Full seeds  :', FULL_SEEDS)


## 2. Preflight fairness

Cell này xác nhận toàn bộ config có `data`, `train` và protocol giống hệt recipe trung tâm; đồng thời build và forward tất cả mô hình trước khi huấn luyện.

In [ ]:
import pandas as pd

from lightweight_hbcc.config import load_config
from tools.run_fair_comparison import config_paths, validate_controlled_configs

fair_configs = validate_controlled_configs(
    DATASET,
    config_paths(DATASET, FULL_MODELS),
)
reference_cfg = fair_configs[FULL_MODELS[0]]

recipe_summary = {
    'protocol': reference_cfg['protocol']['name'],
    'paper reference': reference_cfg['protocol']['reference']['paper'],
    'paper section': reference_cfg['protocol']['reference']['section'],
    'CIFAR adaptation': reference_cfg['protocol']['reference']['adaptation'],
    'epochs': reference_cfg['train']['epochs'],
    'warmup_epochs': reference_cfg['train']['warmup_epochs'],
    'batch_size': reference_cfg['data']['batch_size'],
    'split_seed': reference_cfg['data']['split_seed'],
    'randaugment': reference_cfg['data']['randaugment'],
    'random_erasing': reference_cfg['data']['random_erasing'],
    'label_smoothing': reference_cfg['train']['label_smoothing'],
    'mixup_alpha': reference_cfg['train']['mixup_alpha'],
    'cutmix_alpha': reference_cfg['train']['cutmix_alpha'],
    'cutmix_prob': reference_cfg['train']['cutmix_prob'],
    'kd_alpha': reference_cfg['train']['kd_alpha'],
}
display(pd.DataFrame([recipe_summary]).T.rename(columns={0: 'giá trị'}))
print(f'Preflight đạt: {len(fair_configs)} mô hình dùng cùng recipe.')


## 3. Kiểm tra P-HBCC-2M

Kiểm tra output shape, chuỗi kích thước feature và ngân sách tham số trước khi chạy training.

In [ ]:
import torch

from lightweight_hbcc.models import build_model

phbcc_cfg = load_config(f'configs/fair_comparison/{DATASET}/phbcc_2m.yaml')
model = build_model(phbcc_cfg).eval()
feature_shapes = []
hooks = [
    module.register_forward_hook(
        lambda _module, _inputs, output: feature_shapes.append(tuple(output.shape))
    )
    for module in [model.stem, *model.downsamples]
]
with torch.inference_mode():
    logits = model(torch.randn(2, 3, 32, 32))
for hook in hooks:
    hook.remove()

params_total = sum(p.numel() for p in model.parameters())
params_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
expected_total = 1_953_056 if DATASET == 'cifar10' else 1_974_026
expected_classes = 10 if DATASET == 'cifar10' else 100

assert logits.shape == (2, expected_classes)
assert params_total == expected_total
assert params_total < 2_000_000

display(pd.DataFrame({
    'thuộc tính': ['feature shapes', 'logits', 'params total', 'params trainable'],
    'giá trị': [feature_shapes, tuple(logits.shape), params_total, params_trainable],
}))
del model, logits


## 4. Hàm chạy command

Mọi training đều đi qua `tools/run_fair_comparison.py`, do đó runner sẽ kiểm tra fairness, paired seed, tên protocol và chống ghi đè.

In [ ]:
def powershell_quote(value):
    return "'" + str(value).replace("'", "''") + "'"


def format_powershell(command):
    return '& ' + ' '.join(powershell_quote(item) for item in command)


def run_command(command):
    command = [str(item) for item in command]
    print(format_powershell(command), flush=True)
    return subprocess.run(command, cwd=ROOT, check=True)


## 5. Smoke test

Smoke run dùng ảnh CIFAR thật nhưng chỉ chạy một batch cho train/validation/test. Accuracy của smoke run **không có giá trị nghiên cứu**.

In [ ]:
smoke_command = [
    PYTHON,
    'tools/run_fair_comparison.py',
    '--dataset', DATASET,
    '--models', *SMOKE_MODELS,
    '--seeds', *map(str, SMOKE_SEEDS),
    '--data-root', DATA_ROOT,
    '--output', OUTPUT_ROOT,
    '--smoke',
]
if FORCE_SMOKE:
    smoke_command.append('--force')

if RUN_SMOKE:
    run_command(smoke_command)
else:
    print('Bỏ qua smoke test vì RUN_SMOKE=False.')


## 6. Huấn luyện đầy đủ

Cảnh báo: ma trận tối giản vẫn gồm 5 mô hình × 3 seeds × 200 epochs. Runner tự bỏ qua run đã hoàn tất và khớp metadata, nhưng từ chối thư mục dở dang/không tương thích; không thêm `--force` cho kết quả chính. Các mô hình mở rộng chỉ chạy khi bạn chủ động thêm vào `FULL_MODELS`.

In [ ]:
full_command = [
    PYTHON,
    'tools/run_fair_comparison.py',
    '--dataset', DATASET,
    '--models', *FULL_MODELS,
    '--seeds', *map(str, FULL_SEEDS),
    '--data-root', DATA_ROOT,
    '--output', OUTPUT_ROOT,
    '--benchmark-output', BENCHMARK_ROOT,
]
if RUN_BENCHMARK_AFTER_FULL:
    full_command.append('--benchmark')

if RUN_FULL:
    run_command(full_command)
else:
    print('Chưa chạy full training. Đặt RUN_FULL=True ở cell cấu hình khi sẵn sàng.')
    print(format_powershell(full_command))


## 7. Đọc kết quả

Cell này đọc `test_metrics.json` và `config.yaml` của mọi run. Bảng chính chỉ nhận run canonical có đúng tên protocol và đúng số epoch của recipe đang chọn.

In [ ]:
import yaml

rows = []
param_cache = {}
for metrics_path in sorted(OUTPUT_ROOT.glob('*/test_metrics.json')):
    run_dir = metrics_path.parent
    config_path = run_dir / 'config.yaml'
    if not config_path.exists():
        continue
    cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    if cfg.get('data', {}).get('name') != DATASET:
        continue
    run_name = run_dir.name
    architecture = run_name.split('_seed', 1)[0].replace(f'fair_{DATASET}_', '')
    cache_key = (architecture, cfg['model']['num_classes'])
    if cache_key not in param_cache:
        counted_model = build_model(cfg)
        param_cache[cache_key] = sum(p.numel() for p in counted_model.parameters())
        del counted_model
    rows.append({
        'run': run_name,
        'architecture': architecture,
        'dataset': cfg['data']['name'],
        'seed': cfg['train']['seed'],
        'protocol': cfg.get('protocol', {}).get('name'),
        'canonical': bool(cfg.get('protocol', {}).get('canonical', False)),
        'epochs': cfg['train']['epochs'],
        'params': param_cache[cache_key],
        'test_acc1': metrics.get('test_acc1'),
        'test_acc5': metrics.get('test_acc5'),
    })

results = pd.DataFrame(rows)
if results.empty:
    print('Chưa có test_metrics.json trong', OUTPUT_ROOT)
else:
    display(results.sort_values(['canonical', 'architecture', 'seed'], ascending=[False, True, True]))

benchmark_rows = []
benchmark_dir = BENCHMARK_ROOT / DATASET
for benchmark_path in sorted(benchmark_dir.glob('*.json')):
    record = json.loads(benchmark_path.read_text(encoding='utf-8'))
    run_name = record.get('config_id', benchmark_path.stem)
    architecture = run_name.split('_seed', 1)[0].replace(f'fair_{DATASET}_', '')
    benchmark_rows.append({
        'architecture': architecture,
        'benchmark_config': run_name,
        'device': record.get('device_name', record.get('device')),
        'params_benchmark': record.get('params_total'),
        'macs': record.get('macs'),
        'latency_ms_b1': record.get('latency_ms_b1'),
        'throughput_b1': record.get('throughput_b1'),
        'peak_memory_mb': record.get('peak_memory_mb'),
    })

benchmark_results = pd.DataFrame(benchmark_rows)
if benchmark_results.empty:
    print('Chưa có benchmark JSON trong', benchmark_dir)
else:
    display(benchmark_results.sort_values('architecture'))


## 8. Tổng hợp paired seeds

Cell kiểm tra coverage của **đúng cả 3 paired seeds** cho mọi kiến trúc và báo chênh lệch accuracy ghép cặp P-HBCC-2M − baseline với 95% CI. Nếu thiếu bất kỳ model/seed nào, notebook không tạo bảng xếp hạng chính. Ba seed là mức tối thiểu nên không diễn giải CI thành bằng chứng non-inferiority mạnh.

In [ ]:
import math
import matplotlib.pyplot as plt

summary = pd.DataFrame()
paired_differences = pd.DataFrame()

if results.empty:
    print('Chưa có kết quả để tổng hợp.')
else:
    canonical_protocol = reference_cfg['protocol']['name']
    canonical_epochs = int(reference_cfg['protocol']['effective_epochs'])
    canonical = results[
        results['canonical']
        & results['test_acc1'].notna()
        & results['protocol'].eq(canonical_protocol)
        & results['epochs'].eq(canonical_epochs)
        & results['architecture'].isin(FULL_MODELS)
    ].copy()
    if canonical.empty:
        print(f'Hiện chỉ có smoke/proxy run; chưa có run canonical {canonical_epochs} epoch.')
    else:
        duplicate_counts = canonical.groupby(['architecture', 'seed']).size()
        if duplicate_counts.gt(1).any():
            duplicated = duplicate_counts[duplicate_counts.gt(1)].to_dict()
            raise ValueError(f'Có nhiều canonical run cho cùng architecture/seed: {duplicated}')

        expected_seeds = set(map(int, FULL_SEEDS))
        seed_sets = {}
        coverage_rows = []
        for architecture in FULL_MODELS:
            present = set(
                canonical.loc[canonical['architecture'] == architecture, 'seed']
                .astype(int)
                .tolist()
            )
            seed_sets[architecture] = present
            coverage_rows.append({
                'architecture': architecture,
                'present_seeds': sorted(present),
                'missing_seeds': sorted(expected_seeds - present),
                'extra_seeds': sorted(present - expected_seeds),
                'complete_exact_pairing': present == expected_seeds,
            })

        coverage = pd.DataFrame(coverage_rows)
        display(coverage)
        complete_pairing = bool(coverage['complete_exact_pairing'].all())
        common_seeds = expected_seeds if complete_pairing else set()
        if not complete_pairing:
            print('Cảnh báo: chưa đủ đúng tập FULL_SEEDS cho mọi mô hình; khóa bảng chính.')
        print('Paired seeds dùng để tổng hợp:', sorted(common_seeds))

        if not common_seeds:
            print('Chưa hoàn tất ma trận paired seed; không tạo bảng xếp hạng.')
        else:
            paired = canonical[canonical['seed'].astype(int).isin(common_seeds)].copy()
            pivot = paired.pivot(index='seed', columns='architecture', values='test_acc1').sort_index()
            if pivot[FULL_MODELS].isna().any().any():
                raise RuntimeError('Ma trận paired seed còn ô trống; kiểm tra coverage ở trên.')

            summary = (
                paired.groupby('architecture', as_index=False)
                .agg(
                    seeds=('seed', 'nunique'),
                    mean_acc1=('test_acc1', 'mean'),
                    std_acc1=('test_acc1', 'std'),
                    min_acc1=('test_acc1', 'min'),
                    max_acc1=('test_acc1', 'max'),
                    params=('params', 'first'),
                )
                .sort_values('mean_acc1', ascending=False)
            )
            summary['std_acc1'] = summary['std_acc1'].fillna(0.0)
            summary['params_m'] = summary['params'] / 1e6

            if not benchmark_results.empty:
                benchmark_one = (
                    benchmark_results.sort_values('benchmark_config')
                    .drop_duplicates('architecture', keep='last')
                )
                benchmark_columns = [
                    'architecture', 'device', 'macs', 'latency_ms_b1',
                    'throughput_b1', 'peak_memory_mb',
                ]
                summary = summary.merge(benchmark_one[benchmark_columns], on='architecture', how='left')
            display(summary)

            target = 'phbcc_2m'
            comparison_rows = []
            if target not in pivot.columns:
                print(f'Không có {target} trong paired matrix; bỏ qua bảng chênh lệch.')
            else:
                t95 = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571, 7: 2.447, 8: 2.365, 9: 2.306, 10: 2.262}
                for baseline in FULL_MODELS:
                    if baseline == target or baseline not in pivot.columns:
                        continue
                    differences = pivot[target] - pivot[baseline]
                    n = int(differences.count())
                    mean_delta = float(differences.mean())
                    std_delta = float(differences.std(ddof=1)) if n > 1 else float('nan')
                    half_width = (
                        t95.get(n, 1.96) * std_delta / math.sqrt(n)
                        if n > 1 else float('nan')
                    )
                    comparison_rows.append({
                        'baseline': baseline,
                        'paired_seeds': n,
                        'mean_delta_pp': mean_delta,
                        'std_delta_pp': std_delta,
                        'ci95_low_pp': mean_delta - half_width,
                        'ci95_high_pp': mean_delta + half_width,
                        'phbcc_wins': int((differences > 0).sum()),
                    })
                paired_differences = pd.DataFrame(comparison_rows)
                display(paired_differences.sort_values('mean_delta_pp', ascending=False))

            ax = summary.plot.bar(
                x='architecture',
                y='mean_acc1',
                yerr='std_acc1',
                capsize=4,
                figsize=(11, 5),
                legend=False,
                title=f'{DATASET.upper()} — paired test accuracy (mean ± std)',
            )
            ax.set_ylabel('Top-1 accuracy (%)')
            ax.set_xlabel('Architecture')
            plt.xticks(rotation=35, ha='right')
            plt.tight_layout()
            plt.show()


## Ghi chú khi viết report

1. Chỉ dùng các run `canonical=true` trong bảng so sánh kiến trúc.
2. Không dùng smoke accuracy hoặc `train_acc1` để kết luận.
3. Báo cáo `mean ± standard deviation`, số parameters và latency/MAC cùng accuracy.
4. Giữ `data.split_seed=42`; thay `train.seed` theo paired seeds.
5. Augmentation/order được kiểm soát theo seed, nhưng `adaptive_avg_pool2d_backward_cuda` chưa deterministic tuyệt đối; không tuyên bố GPU runs bitwise-identical.
6. Protocol công bằng chặn `--resume`, vì pipeline hiện chưa phục hồi đầy đủ optimizer/scheduler/RNG state.